# Notebook 03 — Clinical Feature Engineering

## Goal

Build a unified clinical feature table by combining:

- Quantitative MRI measurements
- Radiology report findings

The generated table will be used by the Clinical Decision Support module.

# Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import re

In [2]:
#TEXT_ENCODER = "clinicalbert"
TEXT_ENCODER = "biobert"

In [3]:
REPORT_FILE = Path(
    "/kaggle/input/datasets/mariammohamed1095/reports/reports/nlp/textbrats_dataset_inventory.csv"
)

FUSION_DIR = Path(
    f"/kaggle/input/datasets/mariammohamed1095/models/datasets/processed/fusion/{TEXT_ENCODER}"
)

OUTPUT_DIR = Path(
    "/kaggle/working/datasets/processed/clinical_features"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [4]:
reports = pd.read_csv(REPORT_FILE)

reports.head()

,patient_id,txt_exists,npy_exists,txt_path,npy_path,report,characters,words,lines,embedding_shape
0,BraTS20_Training_001,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...,The lesion area is in the right frontal and pa...,586,91,1,"(1, 128, 768)"
1,BraTS20_Training_002,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...,"The lesion area is in the right frontal lobe, ...",888,139,1,"(1, 128, 768)"
2,BraTS20_Training_003,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...,"The lesion area is in the left frontal lobe, p...",804,128,1,"(1, 128, 768)"
3,BraTS20_Training_004,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...,The lesion area is in the left temporal lobe a...,449,71,1,"(1, 128, 768)"
4,BraTS20_Training_005,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...,The lesion area is in the left parietal lobe r...,606,94,1,"(1, 128, 768)"


In [5]:
reports.shape

(369, 10)

In [6]:
reports = reports[
    [

        "patient_id",

        "report",

    ]

].copy()

reports.head()

,patient_id,report
0,BraTS20_Training_001,The lesion area is in the right frontal and pa...
1,BraTS20_Training_002,"The lesion area is in the right frontal lobe, ..."
2,BraTS20_Training_003,"The lesion area is in the left frontal lobe, p..."
3,BraTS20_Training_004,The lesion area is in the left temporal lobe a...
4,BraTS20_Training_005,The lesion area is in the left parietal lobe r...


In [7]:
train_meta = pd.read_csv(
    FUSION_DIR / "train_metadata.csv"
)

validation_meta = pd.read_csv(
    FUSION_DIR / "validation_metadata.csv"
)

test_meta = pd.read_csv(
    FUSION_DIR / "test_metadata.csv"
)

In [8]:
print(train_meta.shape)

print(validation_meta.shape)

print(test_meta.shape)

(257, 4)
(56, 4)
(56, 4)


In [9]:
train_reports = reports.merge(

    train_meta[
        ["patient_id"]
    ],

    on="patient_id",

    how="inner",

)

validation_reports = reports.merge(

    validation_meta[
        ["patient_id"]
    ],

    on="patient_id",

    how="inner",

)

test_reports = reports.merge(

    test_meta[
        ["patient_id"]
    ],

    on="patient_id",

    how="inner",

)

In [10]:
print("="*60)

print("Train")

print(train_reports.shape)

print("="*60)

print("Validation")

print(validation_reports.shape)

print("="*60)

print("Test")

print(test_reports.shape)

Train
(257, 2)
Validation
(56, 2)
Test
(56, 2)


In [11]:
print(

    train_reports["patient_id"].nunique()

)

print(

    validation_reports["patient_id"].nunique()

)

print(

    test_reports["patient_id"].nunique()

)

257
56
56


In [12]:
print(

    len(

        set(train_reports.patient_id)

        &

        set(validation_reports.patient_id)

    )

)

print(

    len(

        set(train_reports.patient_id)

        &

        set(test_reports.patient_id)

    )

)

print(

    len(

        set(validation_reports.patient_id)

        &

        set(test_reports.patient_id)

    )

)

0
0
0


In [13]:
def clean_report(text):
    """
    Whitespace normalization only — NO lowercasing.

    🔧 FIX: the original version called text.lower() here. This is
    inconsistent with nlp_module/preprocessing.py clean_report(), which
    deliberately does NOT lowercase because BioBERT and ClinicalBERT are
    cased models pretrained on raw clinical text — lowercasing discards
    casing signal they were trained to use.

    For this notebook's keyword matching (ANATOMICAL_REGIONS, LATERALITY,
    RADIOLOGY_FINDINGS) we compare against lowercase keywords and
    lowercase the text at match-time only, keeping the stored
    clean_report consistent with the NLP module.
    """

    if pd.isna(text):
        return ""

    # whitespace normalization only — matches nlp_module/preprocessing.py
    text = str(text)
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    import re as _re
    text = _re.sub(r"\s+", " ", text)

    return text.strip()


In [14]:
for df in [

    train_reports,

    validation_reports,

    test_reports,

]:

    df["clean_report"] = df["report"].apply(
        clean_report
    )

In [15]:
train_reports[
    [

        "patient_id",

        "clean_report",

    ]

].head()

,patient_id,clean_report
0,BraTS20_Training_002,"The lesion area is in the right frontal lobe, ..."
1,BraTS20_Training_003,"The lesion area is in the left frontal lobe, p..."
2,BraTS20_Training_004,The lesion area is in the left temporal lobe a...
3,BraTS20_Training_005,The lesion area is in the left parietal lobe r...
4,BraTS20_Training_007,The lesion area is in the left parietal region...


In [16]:
ANATOMICAL_REGIONS = {

    "frontal": [
        "frontal"
    ],

    "temporal": [
        "temporal"
    ],

    "parietal": [
        "parietal"
    ],

    "occipital": [
        "occipital"
    ],

    "insula": [
        "insula",
        "insular"
    ],

    "thalamus": [
        "thalamus",
        "thalamic"
    ],

    "basal_ganglia": [
        "basal ganglia"
    ],

    "corpus_callosum": [
        "corpus callosum"
    ],

    "ventricle": [
        "ventricle",
        "ventricular"
    ],

}

In [17]:
LATERALITY = {

    "left": [
        "left"
    ],

    "right": [
        "right"
    ],

    "bilateral": [
        "bilateral",
        "both"
    ],

}

In [18]:
RADIOLOGY_FINDINGS = {

    "edema": [
        "edema"
    ],

    "necrosis": [
        "necrosis",
        "necrotic"
    ],

    "enhancement": [
        "enhancement",
        "enhancing"
    ],

    "mass_effect": [
        "mass effect"
    ],

    "midline_shift": [
        "midline shift"
    ],

    "compression": [
        "compression",
        "compressed"
    ],

    "hemorrhage": [
        "hemorrhage",
        "hemorrhagic"
    ],

}

In [19]:
def contains_any(text, keywords):
    """
    Check if any keyword appears in the text (case-insensitive).

    Lowercasing happens here at match-time only — not in clean_report() —
    so the stored report text remains consistent with the NLP module.
    """
    text_lower = text.lower()
    return int(any(keyword in text_lower for keyword in keywords))


In [20]:
def extract_report_features(df):

    rows = []

    for _, row in df.iterrows():

        report = row["clean_report"]

        sample = {

            "patient_id": row["patient_id"]

        }

        # --------------------------
        # Anatomical Regions
        # --------------------------

        for feature, keywords in ANATOMICAL_REGIONS.items():

            sample[feature] = contains_any(

                report,

                keywords,

            )

        # --------------------------
        # Laterality
        # --------------------------

        for feature, keywords in LATERALITY.items():

            sample[feature] = contains_any(

                report,

                keywords,

            )

        # --------------------------
        # Radiology Findings
        # --------------------------

        for feature, keywords in RADIOLOGY_FINDINGS.items():

            sample[feature] = contains_any(

                report,

                keywords,

            )

        # --------------------------
        # Numeric Features
        # --------------------------

        sample["report_length"] = len(report)

        sample["word_count"] = len(

            report.split()

        )

        sample["sentence_count"] = report.count(".")

        sample["lobe_count"] = (

            sample["frontal"]

            + sample["temporal"]

            + sample["parietal"]

            + sample["occipital"]

        )

        rows.append(sample)

    return pd.DataFrame(rows)

In [21]:
train_findings = extract_report_features(train_reports)

validation_findings = extract_report_features(validation_reports)

test_findings = extract_report_features(test_reports)

In [22]:
train_findings.head()

,patient_id,frontal,temporal,parietal,occipital,insula,thalamus,basal_ganglia,corpus_callosum,ventricle,...,necrosis,enhancement,mass_effect,midline_shift,compression,hemorrhage,report_length,word_count,sentence_count,lobe_count
0,BraTS20_Training_002,1,1,1,0,0,0,0,0,1,...,1,0,0,0,1,0,888,139,4,3
1,BraTS20_Training_003,1,1,1,1,0,0,0,0,1,...,1,0,0,0,1,0,804,128,4,4
2,BraTS20_Training_004,0,1,0,0,0,0,0,0,1,...,1,0,0,0,1,0,449,71,4,1
3,BraTS20_Training_005,0,0,1,0,0,0,0,0,1,...,1,0,0,0,1,0,606,94,4,1
4,BraTS20_Training_007,0,0,1,0,0,0,0,0,1,...,1,0,0,0,1,0,666,97,4,1


In [23]:
train_findings.drop(columns=["patient_id"]).sum().sort_values(ascending=False)

report_length      166277
word_count          25053
sentence_count       1041
lobe_count            545
necrosis              257
edema                 257
ventricle             257
compression           257
parietal              204
frontal               187
left                  161
right                 138
temporal               99
occipital              55
bilateral              21
mass_effect             1
basal_ganglia           0
insula                  0
thalamus                0
enhancement             0
corpus_callosum         0
hemorrhage              0
midline_shift           0
dtype: int64

In [24]:
constant_features = []

for column in train_findings.columns:

    if column == "patient_id":
        continue

    if train_findings[column].nunique() == 1:

        constant_features.append(column)

print("Constant features (dropped):", constant_features)

# 🔧 FIX: save the list so it can be reapplied to new data without
# re-running the full notebook (avoids silent column mismatch downstream).
import json as _json
(OUTPUT_DIR / "dropped_constant_features.json").write_text(
    _json.dumps(constant_features)
)


Constant features (dropped): ['insula', 'thalamus', 'basal_ganglia', 'corpus_callosum', 'ventricle', 'edema', 'necrosis', 'enhancement', 'midline_shift', 'compression', 'hemorrhage']


153

In [25]:
train_findings = train_findings.drop(
    columns=constant_features
)

validation_findings = validation_findings.drop(
    columns=constant_features
)

test_findings = test_findings.drop(
    columns=constant_features
)

In [26]:
print(train_findings.columns)

Index(['patient_id', 'frontal', 'temporal', 'parietal', 'occipital', 'left',
       'right', 'bilateral', 'mass_effect', 'report_length', 'word_count',
       'sentence_count', 'lobe_count'],
      dtype='object')


In [27]:
feature_counts = train_findings.drop(
    columns="patient_id"
).sum()

feature_counts

frontal              187
temporal              99
parietal             204
occipital             55
left                 161
right                138
bilateral             21
mass_effect            1
report_length     166277
word_count         25053
sentence_count      1041
lobe_count           545
dtype: int64

In [28]:
rare_features = feature_counts[
    feature_counts < 5
].index.tolist()

print("Rare features (< 5 occurrences in train, dropped):", rare_features)

# 🔧 FIX: save the list so validation/test preprocessing outside this
# notebook always drops exactly the same columns as training did.
import json as _json
(OUTPUT_DIR / "dropped_rare_features.json").write_text(
    _json.dumps(rare_features)
)


Rare features (< 5 occurrences in train, dropped): ['mass_effect']


15

In [29]:
train_findings.drop(
    columns=rare_features,
    inplace=True,
)

validation_findings.drop(
    columns=rare_features,
    inplace=True,
)

test_findings.drop(
    columns=rare_features,
    inplace=True,
)

In [30]:
train_clinical = train_meta.merge(
    train_findings,
    on="patient_id",
)

validation_clinical = validation_meta.merge(
    validation_findings,
    on="patient_id",
)

test_clinical = test_meta.merge(
    test_findings,
    on="patient_id",
)

In [31]:
print(train_clinical.shape)
print(validation_clinical.shape)
print(test_clinical.shape)

display(train_clinical.head())

(257, 15)
(56, 15)
(56, 15)


,patient_id,wt_volume,tc_volume,et_volume,frontal,temporal,parietal,occipital,left,right,bilateral,report_length,word_count,sentence_count,lobe_count
0,BraTS20_Training_002,67008,15709,6549,1,1,1,0,0,1,0,888,139,4,3
1,BraTS20_Training_003,29807,3731,2998,1,1,1,1,1,0,0,804,128,4,4
2,BraTS20_Training_004,103496,26400,15498,0,1,0,0,1,0,0,449,71,4,1
3,BraTS20_Training_005,21963,14410,10786,0,0,1,0,1,1,0,606,94,4,1
4,BraTS20_Training_007,39772,9557,6159,0,0,1,0,1,0,0,666,97,4,1


In [32]:
correlation = train_clinical[
    [
        "report_length",
        "word_count",
        "sentence_count",
    ]
].corr()

print(correlation)

                report_length  word_count  sentence_count
report_length        1.000000    0.979251        0.320099
word_count           0.979251    1.000000        0.311527
sentence_count       0.320099    0.311527        1.000000


In [33]:
columns_to_drop = [
    "report_length"
]

train_clinical.drop(
    columns=columns_to_drop,
    inplace=True,
)

validation_clinical.drop(
    columns=columns_to_drop,
    inplace=True,
)

test_clinical.drop(
    columns=columns_to_drop,
    inplace=True,
)

In [34]:
print(train_clinical.shape)
print(validation_clinical.shape)
print(test_clinical.shape)

(257, 14)
(56, 14)
(56, 14)


In [35]:
train_clinical.to_csv(
    OUTPUT_DIR / "train_clinical_features.csv",
    index=False,
)

validation_clinical.to_csv(
    OUTPUT_DIR / "validation_clinical_features.csv",
    index=False,
)

test_clinical.to_csv(
    OUTPUT_DIR / "test_clinical_features.csv",
    index=False,
)

print("Final clinical feature tables saved successfully.")

Final clinical feature tables saved successfully.


In [36]:
for split in ["train", "validation", "test"]:

    df = pd.read_csv(
        OUTPUT_DIR / f"{split}_clinical_features.csv"
    )

    print("=" * 60)
    print(split.upper())
    print(df.shape)
    print(df.columns.tolist())

TRAIN
(257, 14)
['patient_id', 'wt_volume', 'tc_volume', 'et_volume', 'frontal', 'temporal', 'parietal', 'occipital', 'left', 'right', 'bilateral', 'word_count', 'sentence_count', 'lobe_count']
VALIDATION
(56, 14)
['patient_id', 'wt_volume', 'tc_volume', 'et_volume', 'frontal', 'temporal', 'parietal', 'occipital', 'left', 'right', 'bilateral', 'word_count', 'sentence_count', 'lobe_count']
TEST
(56, 14)
['patient_id', 'wt_volume', 'tc_volume', 'et_volume', 'frontal', 'temporal', 'parietal', 'occipital', 'left', 'right', 'bilateral', 'word_count', 'sentence_count', 'lobe_count']
